# NAICS-2 Classification with RBF SVM

Using 3072-dim embeddings (OpenAI text-embedding-3-large) with an RBF kernel SVM.

Best model from classifier comparison — 69.4% top-1 on 20 NAICS-2 sector classes.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Imports done.")

In [ ]:
csv_path = '../../.ipynb_checkpoints/ExioNAICS_embeddings_large_full.csv'
npy_cache = csv_path.rsplit('.', 1)[0] + '.npy'

df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")

if os.path.exists(npy_cache):
    print(f"\nLoading cached embeddings from {npy_cache}")
    start = time.time()
    X = np.load(npy_cache)
    print(f"Done in {time.time()-start:.1f}s")
else:
    print("\nParsing embeddings...")
    start = time.time()
    X = np.array([json.loads(e) for e in df['embeddings']], dtype=np.float32)
    print(f"Done in {time.time()-start:.1f}s")
    np.save(npy_cache, X)

df['emb'] = list(X)
print(f"Embedding shape: {X.shape}")

In [ ]:
df = df.drop_duplicates(subset=['Company Name']).reset_index(drop=True)
X = normalize(np.array(df['emb'].tolist()))

NAICS2_MAP = {31: 33, 32: 33, 44: 45, 48: 49}
df['NAICS_2_merged'] = df['NAICS_2 Code'].replace(NAICS2_MAP)

le2 = LabelEncoder()
y2 = le2.fit_transform(df['NAICS_2_merged'].astype(str))

print(f"Samples: {len(df)}, Classes: {len(le2.classes_)}")

SEED = 192
X_train, X_val, y2_train, y2_val = train_test_split(
    X, y2, test_size=0.1, random_state=SEED
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

PCA_DIMS = 512
pca = PCA(n_components=PCA_DIMS, random_state=SEED)
X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)
print(f"PCA: 3072 -> {PCA_DIMS} dims ({pca.explained_variance_ratio_.sum():.1%} variance retained)")

## Train RBF SVM

In [ ]:
print("Training RBF SVM...")
start = time.time()
model = SVC(kernel='rbf', C=10.0, class_weight='balanced', probability=True, random_state=SEED)
model.fit(X_train, y2_train)
print(f"Done in {time.time()-start:.1f}s")

# Top-k accuracy
TOP_K = [1, 3, 5]
proba = model.predict_proba(X_val)
for k in TOP_K:
    top_k_preds = np.argsort(proba, axis=1)[:, -k:]
    correct = np.any(top_k_preds == y2_val[:, None], axis=1).sum()
    print(f"  Top-{k}: {correct / len(y2_val):.4f}")

## Per-class Performance

In [ ]:
y_pred = model.predict(X_val)
print(classification_report(y2_val, y_pred, target_names=le2.classes_))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y2_val, y_pred)
cm_df = pd.DataFrame(cm, index=le2.classes_, columns=le2.classes_)
print("Confusion Matrix (rows=true, cols=predicted):")
print(cm_df)